# 04 Business Momentum Weekly Trends
Classify business momentum with rolling windows and threshold experiments.

In [1]:
import warnings
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

warnings.filterwarnings("ignore")
PROJECT_ROOT = Path.cwd().resolve()

if not (PROJECT_ROOT / "utils" / "utils.py").exists():

    if (PROJECT_ROOT / "notebooks" / "utils" / "utils.py").exists():
        PROJECT_ROOT = PROJECT_ROOT / "notebooks"

    elif (PROJECT_ROOT.parent / "utils" / "utils.py").exists():
        PROJECT_ROOT = PROJECT_ROOT.parent

# Add project root to Python path
sys.path.insert(0, str(PROJECT_ROOT))

from utils.utils import ensure_project_dirs, load_raw_dataset, clean_dataset, PROCESSED_DIR, REPORTS_DIR, FIGURES_DIR
from utils.features import engineer_kpis, build_post_feature_sets, aggregate_business_features
from utils.evaluation import regression_metrics, rank_models
from utils.visualization import set_plot_style, save_figure

set_plot_style()
ensure_project_dirs()
RAW_DATA_PATH = Path(r"C:\univ\Data mining\Project\synthetic_generator\synthetic_social_media_posts.csv")
if not RAW_DATA_PATH.exists():
    RAW_DATA_PATH = PROJECT_ROOT / "synthetic_generator" / "synthetic_social_media_posts.csv"
KPI_PATH = PROCESSED_DIR / "kpi_dataset.csv"

c:\Users\hanai\anaconda3\Lib\site-packages\pandas\core\computation\expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.8.4' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
c:\Users\hanai\anaconda3\Lib\site-packages\pandas\core\arrays\masked.py:56: UserWarning: Pandas requires version '1.4.2' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


## Load KPI Dataset and Weekly Aggregation

In [ ]:
if KPI_PATH.exists():
    df = pd.read_csv(KPI_PATH, parse_dates=["post_date"])
else:
    df = engineer_kpis(clean_dataset(load_raw_dataset(RAW_DATA_PATH)))
df.head()

,business_name,sector,followers_count,post_date,posting_hour,day_of_week,month,post_type,caption_text,caption_length,...,is_reel,posting_time_bin,caption_length_bin,hashtags_count_bin,emoji_count_bin,engagement_level,business_size_bin,high_engagement,high_view_rate,high_comment_rate
0,Ramallah Care Pharmacy,pharmacy,13666,2025-12-16,22,Tuesday,12,carousel,صحتك بتهمنا Swipe وشوفوا الباقي. موجودين في Ra...,109,...,0,night,medium,few,none,low,large,0,0,0
1,Nablus Bites,restaurant,12838,2025-10-18,20,Saturday,10,video,اليوم عنا أكل طيب فيديو جديد. احنا جاهزين احجز...,78,...,0,evening,medium,few,none,medium,medium,0,0,0
2,Al Amal Dental,clinic,4652,2025-01-17,21,Friday,1,carousel,صحتكم أولويتنا سوايب وشوفوا التفاصيل. شو رأيكم...,142,...,0,night,long,few,few,low,small,0,0,0
3,Hebron Study Hub,education_center,26289,2025-03-27,20,Thursday,3,reel,استنونا بدورة جديدة شوفوا الريل. زورونا بفرعنا...,90,...,1,evening,medium,few,few,high,large,1,1,1
4,Bethlehem Brew Bar,cafe,2210,2025-11-01,22,Saturday,11,image,أجواء ولا أروع صورة جديدة. أهلا بالناس الحلوة ...,72,...,0,night,medium,few,none,medium,small,0,0,0


## Overall Weekly Trends

In [ ]:
weekly_trends = df.groupby("week", as_index=False).agg(
    total_engagement=("engagement", "sum"),
    avg_engagement_rate=("engagement_rate", "mean"),
    total_likes=("likes_count", "sum"),
    total_comments=("comments_count", "sum"),
    total_views=("views_count", "sum"),
    total_posts=("business_name", "size"),
).sort_values("week")

weekly_trends["engagement_growth"] = (
    weekly_trends["total_engagement"]
    .pct_change()
    .replace([np.inf, -np.inf], np.nan)
    .fillna(0)
)

weekly_trends["trend_class"] = np.where(
    weekly_trends["engagement_growth"] > 0.10,
    "improving",
    np.where(
        weekly_trends["engagement_growth"] < -0.10,
        "declining",
        "stable"
    )
)

weekly_trends.head()

###  Business Weekly Aggregation

In [ ]:
business_weekly = df.groupby(["business_name", "sector", "week"], as_index=False).agg(
    engagement_rate=("engagement_rate", "mean"),
    engagement=("engagement", "mean"),
    posts_count=("business_name", "size"),
).sort_values(["business_name", "week"])

business_weekly.head()

## Rolling Window and Threshold Experiments

In [ ]:
windows = [2,3,4]
thresholds = [0.05,0.10,0.15]
rows, store = [], {}

for w in windows:
    for t in thresholds:
        tmp = business_weekly.copy().copy()
        tmp["rolling_engagement"] = tmp.groupby("business_name")["engagement_rate"].transform(lambda s: s.rolling(w, min_periods=1).mean())
        tmp["growth"] = tmp.groupby("business_name")["rolling_engagement"].pct_change().replace([np.inf,-np.inf], np.nan).fillna(0)
        tmp["trend_class"] = np.where(tmp["growth"] > t, "improving", np.where(tmp["growth"] < -t, "declining", "stable"))
        changes = tmp.groupby("business_name")["trend_class"].nunique().rename("n_states")
        tmp = tmp.merge(changes, on="business_name", how="left")
        tmp["final_class"] = np.where(tmp["n_states"] >= 3, "inconsistent", tmp["trend_class"])
        final = tmp.groupby("business_name", as_index=False).tail(1)
        rows.append({
            "rolling_window": w,
            "growth_threshold": t,
            "stable_ratio": (final["final_class"]=="stable").mean(),
            "inconsistent_ratio": (final["final_class"]=="inconsistent").mean(),
            "interpretability": 1 - abs(t-0.10) - abs(w-3)*0.05,
        })
        store[(w,t)] = tmp

exp = rank_models(pd.DataFrame(rows), higher_is_better_cols=["stable_ratio","interpretability"], lower_is_better_cols=["inconsistent_ratio"])
best = exp.iloc[0]
tmp = store[(int(best["rolling_window"]), float(best["growth_threshold"]))]
business_momentum = tmp.groupby(["business_name","sector"], as_index=False).tail(1)[["business_name","sector","week","rolling_engagement","growth","final_class"]].rename(columns={"final_class":"momentum_class","rolling_engagement":"latest_rolling_engagement_rate","growth":"latest_growth"})
business_weekly_trends= tmp[["business_name","sector","week","engagement_rate","rolling_engagement","growth","final_class"]].rename(columns={"final_class":"trend_class"})

## Save Outputs and Insights

In [ ]:
business_momentum.to_csv(PROCESSED_DIR / "business_momentum.csv", index=False)
weekly_trends.to_csv(PROCESSED_DIR / "weekly_trends.csv", index=False)
business_weekly_trends.to_csv(PROCESSED_DIR / "business_weekly_trends.csv", index=False)

exp.to_csv(REPORTS_DIR / "momentum_experiments.csv", index=False)
display(exp)
display(business_momentum.head(15))
print("Insight: use declining/inconsistent flags for immediate coaching priorities.")

,rolling_window,growth_threshold,stable_ratio,inconsistent_ratio,interpretability,composite_score
0,3,0.10,0.0,1.0,1.00,2.0
1,4,0.10,0.0,1.0,0.95,1.5
2,2,0.10,0.0,1.0,0.95,1.5
3,3,0.05,0.0,1.0,0.95,1.5
4,3,0.15,0.0,1.0,0.95,1.5
5,2,0.05,0.0,1.0,0.90,1.0
6,2,0.15,0.0,1.0,0.90,1.0
7,4,0.05,0.0,1.0,0.90,1.0
8,4,0.15,0.0,1.0,0.90,1.0


,business_name,sector,week,latest_rolling_engagement_rate,latest_growth,momentum_class
16,Al Amal Dental,clinic,52,0.108298,0.176100,inconsistent
35,Al Balad Grill,restaurant,49,0.415372,0.377474,inconsistent
55,Al Hayat Pharmacy,pharmacy,50,0.064392,-0.074065,inconsistent
78,Al Karmel Boutique,store,51,0.123629,0.060389,inconsistent
99,Bethlehem Brew Bar,cafe,51,0.331327,-0.202186,inconsistent
119,Bethlehem Lash Bar,beauty_salon,52,0.065919,0.179754,inconsistent
145,Canaan Threads,store,52,0.073367,-0.315777,inconsistent
167,Gaza Family Pharmacy,pharmacy,52,0.130872,0.332286,inconsistent
196,Gaza Gadget Shop,electronics_store,48,0.128647,0.464168,inconsistent
214,Gaza Glow Salon,beauty_salon,52,0.125752,-0.521901,inconsistent


Insight: use declining/inconsistent flags for immediate coaching priorities.
